# miniGPT — Session 2: The Data Engine

**Ways of Thinking with Data and Computation · Habib University**

---

On Monday we built a die whose faces are words, and then noticed that we had
no idea what the weights should be. Today we go and find out, by counting real
English.

**No new programming concepts today.** Everything below is loops,
dictionaries, string methods and `open()`. What is new is that the data is
real, and real data does not behave.

| Step | What we do |
|---|---|
| 7 | Read the corpus off disk and look at what is actually in it |
| 8 | Clean it — the hard part, and the part with judgment calls in it |
| 9 | Tokenize: text → a list of words |
| 10 | Count: a list of words → `{word: count}` |
| 11 | Feed the counts into Monday's die → the **unigram model** |
| 12 | Ask whether the output means anything |

### Your flow chart

Get it out. Everything below should map onto a box you drew on Monday. Where
it does not, one of you was wrong, and it is worth working out which.

---
## First: bring Monday's code forward

The generator needs `weighted_pick`. Paste your own Session 1 versions in
below — you wrote them, they are yours, and you will need them again on Friday.

In [91]:
import random
import json
import os


# ---------------------------------------------------------------------------
# FROM SESSION 1 -- paste your own working versions in below.
# If yours did not work on Monday, fix it now: everything today depends on it.
# ---------------------------------------------------------------------------

def tally(outcomes):
    """Count how often each outcome appears. Returns {outcome: count}."""
    # your Session 1 code here
    dict ={}
    for x in outcomes:
        dict[x] = dict.get(x, 0) + 1
        
    

    return dict



def weighted_pick(weights):
    """Pick one key from `weights`, in proportion to its value."""
    # your Session 1 code here
    total = sum(weights.values())
    dicttotal = 0
    target = random.random()*total
    for i in weights:
        dicttotal+=weights[i]
        if target< dicttotal: 
            return i 


# Quick check before you go any further. With these weights, "b" should come
# up roughly three times as often as "a", and "c" roughly twice as often.
test = []
for i in range(3000):
    test.append(weighted_pick({"a": 1, "b": 3, "c": 2}))
print(tally(test))

{'c': 1040, 'b': 1469, 'a': 491}


---
## 7. Reading the data

The data is given in the "Data" page for this case study on Canvas.


`os.listdir(folder)` gives the file names in a folder as a list of strings.
That is the only new function today.

Following code to download the dataset from hugging face

**`encoding="utf-8"` is not optional.** These files contain characters that
are not plain ASCII, and if you leave the encoding to chance the read will
either fail or quietly give you the wrong characters.

In [92]:
CORPUS_DIR = "corpus"


def list_corpus_files(folder):
    """Return a sorted list of the .txt file names in `folder`."""
    names = []
    for name in os.listdir(folder):
        names.append(name)
    names.sort()          # sorted so that everyone's results match
    return names


def read_text_file(path):
    """Return the whole contents of a text file as one string."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


files = list_corpus_files(CORPUS_DIR)
print(len(files), "files:", files)

first = read_text_file(os.path.join(CORPUS_DIR, files[0]))
print("characters in the first file:", len(first))

10 files: ['pg1342.txt', 'pg2465.txt', 'pg2554.txt', 'pg2641.txt', 'pg2701.txt', 'pg3268.txt', 'pg34413.txt', 'pg59828.txt', 'pg65238.txt', 'soda_dialogue_10000.csv']
characters in the first file: 748167


### Big question — what is actually in this file?

Do not take anyone's word for it. Print the beginning and the end and read
them.

In [93]:
print("----- FIRST 1200 CHARACTERS -----")
print(first[:1200])
print()
print("----- LAST 900 CHARACTERS -----")
print(first[-900:])

----- FIRST 1200 CHARACTERS -----
The Project Gutenberg eBook of Pride and Prejudice
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Pride and Prejudice

Author: Jane Austen


        
Release date: June 1, 1998 [eBook #1342]
                Most recently updated: February 10, 2026

Language: English

Other information and formats: www.gutenberg.org/ebooks/1342

Credits: Chuck Greif and the Online Distributed Proofreading Team at http://www.pgdp.net (This file was produced from images available at The Internet Archive)


*** START OF THE PROJECT GUTENBERG EBOOK PRIDE AND PREJUDICE ***


---
## 8. Cleaning the data

The intellectual heart of the session. Budget more time for this than you
think it needs — everyone always underestimates cleaning, in this classroom
and in industry.

There are two separate jobs, and it helps to keep them apart:

**8a. Throw away the parts of the file that are not the book.**
Project Gutenberg is kind to us here: every file has a marker line.

```
*** START OF THE PROJECT GUTENBERG EBOOK PRIDE AND PREJUDICE ***
...the actual novel...
*** END OF THE PROJECT GUTENBERG EBOOK PRIDE AND PREJUDICE ***
```

Keep only what is between them. Note that the title is *inside* the marker
line, so you cannot match the whole line — you match the part that is the
same in every file.

**8b. Normalise the words that remain.** `The`, `the` and `“the` should all
be counted as the same word. This is where the judgment calls live.

In [94]:
START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"


def strip_boilerplate(text):
    """Return only the part of `text` between the Gutenberg marker lines.

    YOUR TASK
    ---------
      1. Split the text into lines: `text.split("\n")`.
      2. Walk through the lines keeping a True/False variable that records
         whether you are currently inside the book. It starts False.
      3. When a line starts with START_MARKER, switch it to True and move on
         to the next line -- the marker line itself is not part of the book.
         (Use `line.startswith(START_MARKER)`, not `line == START_MARKER`:
         the book's title is appended to the marker, so the lines are not
         identical between files.)
      4. When a line starts with END_MARKER, switch it back to False.
      5. Collect every line you see while the variable is True.
      6. Join the collected lines back into one string, putting the newline
         characters back as you go.

    Also: if you never saw a start marker, print a warning and return the
    original text unchanged. A cleaning function that silently returns an
    empty string when its assumption breaks is far more dangerous than one
    that complains -- you would not notice until your word counts were
    mysteriously tiny.
    """
    # your code here
    list=[]
    list1=text.split("\n")
    flag=False
    count = 0
    for line in list1:
        if line.startswith(START_MARKER):
            flag=True
            count +=1
        elif line.startswith(END_MARKER):
            flag = False
        elif flag==True:
            # line.strip()
            # if "qwertyuiopasdfghjklzxcvbnmQWERTYUIOPASDFGHJKLZXCVBNM"in line:
            list.append(line)
    # print(list)
    if count == 0:
        print("Warning")
        return text         
    else:
        return("\n".join(list)  )

body = strip_boilerplate(first)
print("before:", len(first), "characters")
print("after: ", len(body), "characters")
print("removed:", len(first) - len(body), "characters of boilerplate")
print()
print(body[:400])
# You should have removed roughly 20,000 characters, and the text should now
# begin with the novel itself rather than with a licence notice.

before: 748167 characters
after:  728749 characters
removed: 19418 characters of boilerplate





                            [Illustration:

                             GEORGE ALLEN
                               PUBLISHER

                        156 CHARING CROSS ROAD
                                LONDON

                             RUSKIN HOUSE
                                   ]

                            [Illustration:

               _Reading Jane’s Letters._      _Chap 34._



In [95]:
import pytest

# Runs the strip_boilerplate tests in test_lecture2.py against the function above.
result = pytest.main(["-q", "-p", "no:cacheprovider", "-k", "strip_boilerplate", "test_lecture2.py"])

.....                                                                    [100%]
5 passed, 13 deselected in 0.02s


### 8b. Normalising words — where the judgment calls are

Now the messy part. Look at what a naive `text.split()` actually hands you:

```
'“It      'universally,'    'Mr.'    'mother-in-law'    '_Netherfield_'
'don’t'   'Bennet,”'        'www.gutenberg.org'         '1813'
```

Every one of those is a decision.

| Raw | Options | Consequence |
|---|---|---|
| `“It` | strip the quote mark | otherwise `“It` and `It` are different words |
| `Mr.` | strip the full stop | otherwise `Mr.` and `Mr` split the count |
| `mother-in-law` | one word, or three? | changes what the model can generate |
| `don’t` | keep, or split into `do` + `n’t`? | English contractions are genuinely two words |
| `1813` | keep numbers, or drop them? | numbers are rarely useful in a word model |
| `_Netherfield_` | strip the underscores | Gutenberg marks italics with them |

**One trap you will hit and will not expect.** Type an apostrophe on your
keyboard and you get `'` — ASCII character 39. Now count them in the corpus:

```python
first.count("'")   # ->  0
first.count("’")   # ->  767
```

**There is not a single ASCII apostrophe in the file.** Every one of them is
`’`, the "right single quotation mark", a different character that merely
looks the same. If you write `if ch == "'"` your rule will never fire even
though the apostrophes are clearly right there on screen. This is not a
contrived example — it is one of the most common real bugs in text processing,
and it is invisible until you go and check.

**Our rules** (defensible, not uniquely correct — argue with them):

1. Lowercase everything.
2. Split hyphenated compounds into their parts.
3. Keep letters and apostrophes (both kinds); drop everything else.
4. Drop anything that ends up empty — pure punctuation, page numbers.

In [96]:
APOSTROPHES = "'\u2019\u2018\u00b4"        # ASCII ', curly \u2019 \u2018, acute \u00b4


def clean_word(word):
    # x = "123456789!@#$%^&*()"


    """Normalise one raw token into a comparable word (possibly empty).

    YOUR TASK
    ---------
    Build up a result string one character at a time:
      - if the character is a letter (`ch.isalpha()`), add its lowercase form
      - if it is any of the apostrophes in APOSTROPHES, add a plain ASCII "'"
        so that don't and don\u2019t become the same word
      - otherwise skip it
    Finally, strip any apostrophes left at the very start or end of the
    result -- those came from quotation marks, not from contractions.
    (`result.strip("'")` does exactly that.)

    Return the result. It is allowed to come back empty: "1813" and "--"
    contain no letters at all, and the caller deals with that.
    """
    # your code here
    str1=""
    for i in word:
      # print(i.isalpha())

      if i in APOSTROPHES:
        str1+="'"
      elif i.isalpha():
        str1+=i
    # print(str1)
    str1 =str1.strip("'")
    # print(str1)
    return str1.lower()


def tokenize(text):
    """Turn a block of text into a list of cleaned words.
    
    YOUR TASK
    ---------
      1. Split the text on whitespace with `text.split()`.
      2. Each chunk may still be a hyphenated compound, so split it again on
         "-" so that mother-in-law becomes mother, in, law.
      3. Run every piece through `clean_word`.
      4. Keep the non-empty results, in order, in a list. Order matters
         enormously -- next session we predict each word from the one before it,
         and a shuffled list would destroy that.

    Return the list.
    """
    # your code here
    output = []
    list2 = text.split()
    list3=[]
    for j in range(len(list2)):
       list3.extend(list2[j].split("-"))
    for i in range(len(list3)):
       y=clean_word(str(list3[i]))
       if y != '':
        output.append(y)
      
    return output
          
        



for raw in ["\u201cIt", "Mr.", "mother-in-law", "don\u2019t", "1813",
            "_Netherfield_", "www.gutenberg.org", "--", "\u2019tis"]:
    print("%-20r -> %r" % (raw, tokenize(raw)))

# Expected:
#   '\u201cIt'              -> ['it']
#   'Mr.'                   -> ['mr']
#   'mother-in-law'         -> ['mother', 'in', 'law']
#   'don\u2019t'               -> ["don't"]
#   '1813'                  -> []
#   '_Netherfield_'         -> ['netherfield']
#   'www.gutenberg.org'     -> ['wwwgutenbergorg']     <-- look at this one
#   '--'                    -> []
#   '\u2019tis'                -> ['tis']

'“It'                -> ['it']
'Mr.'                -> ['mr']
'mother-in-law'      -> ['mother', 'in', 'law']
'don’t'              -> ["don't"]
'1813'               -> []
'_Netherfield_'      -> ['netherfield']
'www.gutenberg.org'  -> ['wwwgutenbergorg']
'--'                 -> []
'’tis'               -> ['tis']


### Check your tokenizer with tests

The cell below runs **pytest** on your `tokenize`. The test cases live in
`test_lecture2.py`, next to this notebook. Each one gives a raw piece of text
and the list of words we expect back. Every case that fails shows you which
input broke and what your function returned instead.

If you disagree with an expected answer (maybe you decided `mother-in-law`
should stay one word), you are allowed to change it in `test_lecture2.py`.
Be ready to defend the change in the reflection. *A test is your rules
written down.*

In [97]:
import pytest

# Runs the tokenize tests in test_lecture2.py against the tokenize defined above.
result = pytest.main(["-q", "-p", "no:cacheprovider", "-k", "tokenize", "test_lecture2.py"])

.............                                                            [100%]
13 passed, 5 deselected in 0.04s


---
## 9. Tokenization — and what the word means

You have just written a **tokenizer**. That is the real term, and it is the
same job that sits at the front of every language model ever built.

The list of *unique* tokens is your vocabulary — the set of faces on
first session's die.

**Big question.** Real tokenizers do not split on spaces. They split *inside*
words: `walking` → `walk` + `ing`, and rare words get broken into pieces.
Why might that be smarter than what we just did?

Now run the whole corpus through. This takes a few seconds because the data is huge!

In [98]:
# CORPUS_DIR = "corpus"
def load_corpus(folder):
    """Read every book in `folder`, strip boilerplate, return one token list."""
    all_tokens = []
    for name in list_corpus_files(folder):
        raw = read_text_file(os.path.join(folder, name))
        body = strip_boilerplate(raw)
        tokens = tokenize(body)
        print("   %-14s %8d tokens" % (name, len(tokens)))
        all_tokens += tokens
    return all_tokens


print("Reading the corpus...")
TOKENS = load_corpus(CORPUS_DIR)
print()
print("total tokens:", len(TOKENS))
print("first 25:", TOKENS[:25])

Reading the corpus...
   pg1342.txt       127853 tokens
   pg2465.txt        24126 tokens
   pg2554.txt       205050 tokens
   pg2641.txt        66885 tokens
   pg2701.txt       214986 tokens
   pg3268.txt       291035 tokens
   pg34413.txt       33826 tokens
   pg59828.txt      442204 tokens
   pg65238.txt       74772 tokens
Warning
   soda_dialogue_10000.csv  1156182 tokens

total tokens: 2636919
first 25: ['illustration', 'george', 'allen', 'publisher', 'charing', 'cross', 'road', 'london', 'ruskin', 'house', 'illustration', 'reading', "jane's", 'letters', 'chap', 'pride', 'and', 'prejudice', 'by', 'jane', 'austen', 'with', 'a', 'preface', 'by']


---
## 10. Counting words

**Estimate first — write your answers down before running the next cell.**

1. How many words in total are in the corpus?
2. How many *different* words? (The vocabulary size.)
3. What is the most common word, and roughly what share of the text is it?

Being wrong by a factor of ten is the useful outcome here. Most people badly
overestimate the vocabulary and badly underestimate how dominant `the` is.

In [99]:
def count_words(tokens):
    """Return {word: how many times it appears}.

    YOUR TASK
    ---------
    This is `tally` from Session 1 under a different name -- you already know
    how to do it. Write it out again anyway rather than calling `tally`: in
    Session 3 this function grows into something considerably less obvious,
    and it helps to have it standing on its own.
    """
    # your code here
    dict ={}
    for x in tokens:
        dict[x] = dict.get(x, 0) + 1
    return dict



WORD_COUNTS = count_words(TOKENS)

print("total words:      ", len(TOKENS))
print("different words:  ", len(WORD_COUNTS))
print("the most common:  ", WORD_COUNTS["the"], "occurrences of 'the'")
print("that is           ", round(100 * WORD_COUNTS["the"] / len(TOKENS), 1),
      "% of the entire corpus")

total words:       2636919
different words:   66012
the most common:   100948 occurrences of 'the'
that is            3.8 % of the entire corpus


### The top ten

We want the ten most common words. Sorting a dictionary by its values is a
one-line trick in Python, and we are not going to use it — write the search
yourself, so that you know what it costs.

The algorithm: find the largest, set it aside, find the largest of what is
left, and repeat ten times.

In [100]:
def top_n(counts, n):
    """Return the n highest-counting entries as a list of (word, count).

    YOUR TASK
    ---------
    Repeated selection. Do it n times:
      - scan every word in `counts` and remember the one with the largest
        count that you have not already taken
      - record it in your result list as a (word, count) pair
      - mark it as taken so the next scan skips it

    Use a dictionary to remember what you have taken -- `if word not in taken`
    is fast on a dictionary and slow on a list, and with 40,000 words that
    difference is noticeable.

    Guard the case where `counts` has fewer than n entries: if a scan finds
    nothing at all, stop early rather than appending None.

    Think about the cost: you scan all 40,000 words, ten times over. That is
    400,000 comparisons to find ten words. Is that reasonable? What if you
    wanted the top 1,000?
    """
    # your code here
    result = []
    taken = {}                

    for i in range(n):
        best_word = None
        best_count = 0        
        for word in counts:
            if word not in taken and counts[word] > best_count:
                best_word = word
                best_count = counts[word]  
        if best_word is None:   
            break
        result.append((best_word, best_count)) 
        taken[best_word] = True

    return result
# top_n(WORD_COUNTS, 3)
print("rank  word          count      share")
rank = 0
for word, count in top_n(WORD_COUNTS, 10):
    rank += 1
    print("%3d   %-12s %8d   %5.2f%%"
          % (rank, word, count, 100 * count / len(TOKENS)))

rank  word          count      share
  1   the            100948    3.83%
  2   to              85154    3.23%
  3   and             71227    2.70%
  4   i               67536    2.56%
  5   you             60034    2.28%
  6   a               54545    2.07%
  7   of              53806    2.04%
  8   that            37789    1.43%
  9   it              37645    1.43%
 10   in              32105    1.22%


### Big question — why are the top ten so boring?

Not one of them carries any meaning on its own: *the, and, to, of, a, i, in,
that, he, you*. Together they are about a fifth of everything ever written in
these nine books.

This is a real and universal linguistic fact, and you just discovered it from
your own data rather than being told it. It holds in every corpus in every
language: a small number of function words dominate, and the words that carry
the meaning are individually rare.

Check the other end of the distribution too — how many words appear exactly
once? That number is going to matter a great deal on Friday.

In [101]:
once = 0
twice_or_fewer = 0
for word in WORD_COUNTS:
    if WORD_COUNTS[word] == 1:
        once += 1
    if WORD_COUNTS[word] <= 2:
        twice_or_fewer += 1

print("words appearing exactly once:", once,
      "(%.1f%% of the vocabulary)" % (100 * once / len(WORD_COUNTS)))
print("words appearing twice or less:", twice_or_fewer,
      "(%.1f%% of the vocabulary)" % (100 * twice_or_fewer / len(WORD_COUNTS)))

words appearing exactly once: 34225 (51.8% of the vocabulary)
words appearing twice or less: 43064 (65.2% of the vocabulary)


---
## 11. Putting it together — the unigram model

Remember the design decision from Monday: **one program builds the data, a
different program generates the text.**

So we do it in two parts, and the boundary between them is a file on disk.

### Part one: the data engine writes the counts out

`json.dump` writes a dictionary to a file as text; `json.load` reads it back.
That is all we use it for. Open `word_count.json` in a text editor afterwards
and look at it — it is just your dictionary, written down.

In [102]:
os.makedirs("models", exist_ok=True)

with open(os.path.join("models", "word_count.json"), "w", encoding="utf-8") as f:
    json.dump(WORD_COUNTS, f)

size = os.path.getsize(os.path.join("models", "word_count.json"))
print("wrote models/word_count.json")
print("size: %.1f KB" % (size / 1024))
print()
print("Remember this number. On Friday it becomes 60 MB.")

wrote models/word_count.json
size: 1010.8 KB

Remember this number. On Friday it becomes 60 MB.


### Part two: the generator loads the counts and rolls the die

From here on, pretend the cells above do not exist. This is a separate
program. All it knows is that there is a file with a dictionary in it.

And now the payoff of the whole session: **the counts are the weights.**
`weighted_pick` has not changed since Monday, and it does not need to. It was
always going to work; it was only ever waiting for good numbers.

In [103]:
def load_model(path):
    """Load a counts dictionary that the data engine wrote out."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def make_sentence(weights, length):
    """Roll the word die `length` times and join the results with spaces."""
    words = []
    for i in range(length):
        words.append(weighted_pick(weights))

    sentence = ""
    for i in range(len(words)):
        if i > 0:
            sentence += " "
        sentence += words[i]
    return sentence


MODEL = load_model(os.path.join("models", "word_count.json"))
print("loaded a die with", len(MODEL), "faces")
print()

for i in range(5):
    print("-", make_sentence(MODEL, 14))

loaded a die with 66012 faces

- will glory pounds on realize the her on means leisure descried when you like
- keep alone with believe to say the tale why so elizabeth's to finish i
- rather things the make hast soul light picture going photography thought and end sir
- again itgreat should strength was what leaves strengthened like killed much lover and sounds
- you tell ear to moan this and words out i'll in sure a sincerity
